# 05 · Hand-off package for Krishna and Rucha

Collects the final models, histories, the split and a `HANDOFF.md` that explains how to load and use them.
**No test-set results are included** - the test split has not been used for training, early stopping, or experiment selection, so Rucha's final evaluation is clean.

In [ ]:
# ============ SETUP - run this first after every fresh Colab runtime ============
import os, sys, subprocess
from pathlib import Path

REPO_URL = ""   # e.g. "https://github.com/<your-username>/fruit-freshness-classifier.git"
                # leave "" if you uploaded/unzipped the project folder yourself

IN_COLAB = "google.colab" in sys.modules
PROJECT_DIR = Path("/content/fruit-freshness-classifier") if IN_COLAB else Path.cwd().parent
if not (PROJECT_DIR / "src").exists():
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    else:
        raise SystemExit(f"Project folder not found at {PROJECT_DIR}. Set REPO_URL above or upload the project there.")
elif REPO_URL and IN_COLAB:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

if os.getenv("FRUIT_SKIP_PIP") != "1":     # only kagglehub is needed; Colab already has TensorFlow etc.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements-colab.txt")], check=True)
sys.path.insert(0, str(PROJECT_DIR / "src"))

import json
import matplotlib.pyplot as plt
import pandas as pd
from fruitfresh import config
from fruitfresh.colab import setup_workspace, gpu_summary

WORK_DIR = setup_workspace()      # local folder <project>/workspace (nothing goes to Google Drive)
SEED = config.SEED
print(gpu_summary())
print("Project folder :", PROJECT_DIR)
print("Work folder    :", WORK_DIR, "(models, histories, results are saved here)")

In [ ]:
# Only needed after a Colab runtime reset: put your downloaded backup zip back into the workspace.
# (On your own PC the workspace/ folder is still there, so leave this False.)
RESTORE = False
if RESTORE:
    from fruitfresh.colab import restore_workspace
    restore_workspace(WORK_DIR)          # Colab: opens an upload dialog for fruit_workspace_backup.zip

In [ ]:
from fruitfresh.data_index import get_dataset_root
from fruitfresh.splits import load_split

DATASET_ROOT = get_dataset_root()
SPLIT_DIR = WORK_DIR / "data" / "splits"
if not SPLIT_DIR.exists():
    SPLIT_DIR = PROJECT_DIR / "data" / "splits"      # the committed copy of the team split
train_df, val_df, _ = load_split(SPLIT_DIR)           # the TEST split is deliberately ignored (`_`)
print(f"train {len(train_df)} images / {train_df.source_id.nunique()} photos | "
      f"validation {len(val_df)} images / {val_df.source_id.nunique()} photos")
print("Split folder:", SPLIT_DIR)

In [ ]:
from fruitfresh.experiments import pick_best
from fruitfresh.train import run_dir_for
table = pd.read_csv(WORK_DIR / "experiments.csv")
mob = table[table["run"] != "baseline_cnn"]
best_run = pick_best(mob)["run"]
print("Final MobileNetV2 run (best on validation):", best_run)
display(table)

In [ ]:
# make sure the baseline has validation metrics stored as well
from fruitfresh.evaluate import evaluate_validation
for run in ("baseline_cnn", best_run):
    evaluate_validation(run_dir_for(WORK_DIR, run, SEED), val_df, DATASET_ROOT)

from fruitfresh.handoff import build_handoff
HANDOFF_DIR = WORK_DIR / "handoff"
build_handoff(WORK_DIR, {"baseline_cnn": "baseline_cnn", "mobilenet_v2_final": best_run}, SEED,
              HANDOFF_DIR, SPLIT_DIR, WORK_DIR / "experiments.csv")
print((HANDOFF_DIR / "HANDOFF.md").read_text())

In [ ]:
import shutil
archive = shutil.make_archive(str(WORK_DIR / "handoff_onkar"), "zip", HANDOFF_DIR)
print("Share this file:", archive)
# small files worth committing to GitHub (models are large, keep them out of git)
res = PROJECT_DIR / "results" / "onkar"; res.mkdir(parents=True, exist_ok=True)
for alias in ("baseline_cnn", "mobilenet_v2_final"):
    for f in ("history.csv", "train_info.json"):
        shutil.copy(HANDOFF_DIR / "runs" / alias / f, res / f"{alias}_{f}")
shutil.copy(HANDOFF_DIR / "HANDOFF.md", res / "HANDOFF.md")
print("Copied small result files to", res)

## Final checklist
- [ ] `handoff_onkar.zip` is saved and shared with Krishna and Rucha
- [ ] `results/` files and `data/splits/` are committed; **models (`*.keras`) are NOT committed** (`.gitignore`)
- [ ] you can answer in the viva: *why two stages? why is BatchNorm frozen? why a time-block split? why was the test set not used?*

## Backup to your own computer
Colab erases its disk when the runtime resets. Run the next cell once you are done (or before closing).

In [ ]:
# Save everything to YOUR computer. In Colab this downloads a zip through the browser, because the
# runtime disk is erased on reset. On your own PC it just writes the zip next to workspace/.
from fruitfresh.colab import backup_workspace
backup_workspace(WORK_DIR)